In [1]:
import pandas as pd, numpy as np, textstat, json, time 
import plotly_express as px
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from sentence_transformers import SentenceTransformer, util

In [4]:
from evaluate import load

In [7]:
bertscore = load("bertscore")
predictions = ["hello there", "general kenobi"]
references = ["hello there", "general kenobi"]
results = bertscore.compute(predictions = predictions, references = references, lang = "en")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1153.89it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
results

{'precision': [1.0000001192092896, 0.9999999403953552],
 'recall': [1.0000001192092896, 0.9999999403953552],
 'f1': [1.0000001192092896, 0.9999999403953552],
 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.0.0)'}

#### Readability

In [ ]:
# Import JSON XAIQB answers
with open("llm_explanations/xaiqb_human_verbose.json") as f:
    xaiqb_human = json.load(f)

with open("llm_explanations/xaiqb_gemini_1.1_v2.json") as f:
    xaiqb_gemini = json.load(f)

with open("llm_explanations/xaiqb_chatgpt_1.1_v2.json") as f:
    xaiqb_chatgpt = json.load(f)

with open("llm_explanations/xaiqb_opus_1.1_v2.json") as f:
    xaiqb_claude = json.load(f)

# with open("xaiqb_chatgpt.json") as f:
#     xaiqb_chatgpt = json.load(f)

# with open("xaiqb_gemini.json") as f:
#     xaiqb_gemini = json.load(f)

# with open("xaiqb_claude.json") as f:
#     xaiqb_claude = json.load(f)

# with open("xaiqb_mistral.json") as f:
#     xaiqb_mistral = json.load(f)

# with open("xaiqb_llama.json") as f:
#     xaiqb_llama = json.load(f)

In [259]:
# https://readabilityformulas.com/
    # https://pypi.org/project/py-readability-metrics/
    
    # if textstat.textstat.lexicon_count(text) >= 100:
    #     r = Readability(text)

    #     scores = {
    #         "text": text,
    #         # "stats": r.statistics(),
    #         "flesch_kincaid": {
    #             "score": r.flesch_kincaid().score, 
    #             "grade_level": r.flesch_kincaid().grade_level,
    #         },
    #         "flesch_reading": {
    #             "score": r.flesch().score,
    #             "ease": r.flesch().ease, 
    #             "grade_level": r.flesch().grade_levels,
    #         },
    #         "dale_chall": {
    #             "score": r.dale_chall().score,
    #             "grade_level": r.dale_chall().grade_levels,
    #         },
    #         "ari": {
    #             "score": r.ari().score,
    #             "grade_level": r.ari().grade_levels,
    #             "ages": r.ari().ages
    #         },
    #         "coleman_liau": {
    #             "score": r.coleman_liau().score, 
    #             "grade_level": r.coleman_liau().grade_level
    #         },
    #         "gunning_fog": {
    #             "score": r.gunning_fog().score, 
    #             "grade_level": r.gunning_fog().grade_level
    #         },
    #         "smog": {
    #             "score": r.smog().score, 
    #             "grade_level": r.smog().grade_level
    #         },
    #         "spache": {
    #             "score": r.spache().score, 
    #             "grade_level": r.spache().grade_level
    #         },
    #         "linsear_write": {
    #             "score": r.linsear_write().score, 
    #             "grade_level": r.linsear_write().grade_level
    #         }
    #     }
    # else:

# pd.DataFrame(scores).drop(["text"], axis = 1).drop(["ease", "ages"], axis = 0) if df else scores 

In [260]:
def readabililty_scores(text):
    
    scores = {

        "flesch_reading_ease":          round(textstat.flesch_reading_ease(text), 3),
        "smog_index":                   round(textstat.smog_index(text), 3),
        "coleman_liau_index":           round(textstat.coleman_liau_index(text), 3),
        "dale_chall_readability_score": round(textstat.dale_chall_readability_score(text), 3),
        
        "text_standard":                textstat.text_standard(text),
        "difficult_words":              round(textstat.difficult_words(text), 3),
        
        # Ignore/irrelevant
        "flesch_kincaid_grade":         round(textstat.flesch_kincaid_grade(text), 3),
        "automated_readability_index":  round(textstat.automated_readability_index(text), 3),
        "linsear_write_formula":        round(textstat.linsear_write_formula(text), 3),
        "gunning_fog":                  round(textstat.gunning_fog(text), 3),
        "spache_readability":           round(textstat.spache_readability(text), 3)
        }

    return scores

In [261]:
# answers_dict = {}
for name, data in zip(["human", "claude", "chatgpt", "gemini"], [xaiqb_human, xaiqb_claude, xaiqb_chatgpt, xaiqb_gemini]):
    answers = []
    for category in data["questions"].keys():
        for question in data["questions"][category].keys():
            answers += [data["questions"][category][question]["answer"]]
    print(name)
    print(readabililty_scores(" ".join(answers)))
    # answers_dict.update({name: " ".join(answers)})

# answers_dict

human
{'flesch_reading_ease': 33.795, 'smog_index': 14.076, 'coleman_liau_index': 14.37, 'dale_chall_readability_score': 12.848, 'text_standard': '12th and 13th grade', 'difficult_words': 336, 'flesch_kincaid_grade': 12.373, 'automated_readability_index': 13.313, 'linsear_write_formula': 8.25, 'gunning_fog': 15.493, 'spache_readability': 6.718}
claude
{'flesch_reading_ease': 2.36, 'smog_index': 19.75, 'coleman_liau_index': 18.438, 'dale_chall_readability_score': 14.961, 'text_standard': '19th and 20th grade', 'difficult_words': 655, 'flesch_kincaid_grade': 20.207, 'automated_readability_index': 23.222, 'linsear_write_formula': 38.5, 'gunning_fog': 23.11, 'spache_readability': 9.246}
chatgpt
{'flesch_reading_ease': 17.744, 'smog_index': 14.535, 'coleman_liau_index': 18.21, 'dale_chall_readability_score': 13.743, 'text_standard': '13th and 14th grade', 'difficult_words': 308, 'flesch_kincaid_grade': 14.347, 'automated_readability_index': 15.783, 'linsear_write_formula': 8.75, 'gunning_fo

In [262]:
# Define function to calculate readability scores for all questions
def get_scores(x = xaiqb_human):
    scores_dict = {}
    for category in x["questions"].keys():
        scores_dict.update({category: {}})
        for question, answer in x["questions"][category].items():
            scores = readabililty_scores(answer["answer"])
            scores_dict[category].update({question: scores})
    return scores_dict

In [263]:
scores_human = get_scores(x = xaiqb_human)
scores_chatgpt = get_scores(x = xaiqb_chatgpt)
scores_gemini = get_scores(x = xaiqb_gemini)
scores_claude = get_scores(x = xaiqb_claude)

In [264]:
def pull_scores(x = scores_human, category = "Data", metric = "flesch_reading_ease"):
    scores = []
    for score in x[category].values():
        scores += [score[metric]]
    return scores

In [265]:
to_plot = []
for category in scores_human.keys():
    for metric in [
        "flesch_reading_ease", 
        "smog_index",
        "coleman_liau_index",
        "dale_chall_readability_score",

        "text_standard",
        "difficult_words",

        "flesch_kincaid_grade",
        "automated_readability_index",
        "linsear_write_formula",
        "gunning_fog", 
        "spache_readability"
        ]:
        to_plot += [pd.DataFrame({
            "category": category,
            "metric": metric,
            "Human": pull_scores(x = scores_human, category = category, metric = metric),
            "ChatGPT 5": pull_scores(x = scores_chatgpt, category = category, metric = metric),
            "Gemini 2.5 Pro": pull_scores(x = scores_gemini, category = category, metric = metric),
            "Opus 4.5": pull_scores(x = scores_claude, category = category, metric = metric),
        })]

to_plot = pd.concat(to_plot).melt(id_vars = ["category", "metric"], var_name = "Respondent", value_name = "Score")
to_plot.loc[to_plot["category"] == "How (global model-wide explanation)", "category"] = "How (Global)"
to_plot.loc[to_plot["category"] == "UI", "category"] = "User Interface"

In [266]:
colour_mapping = {"Human": px.colors.qualitative.G10[3], 
                  "ChatGPT 5": px.colors.qualitative.G10[1],
                  "Gemini 2.5 Pro": px.colors.qualitative.G10[0],
                  "Opus 4.5": px.colors.qualitative.G10[2]}

In [267]:
# Higher score = Better Readability
# [Flesch Reading Ease]

# Lower score = Better Readability
# [SMOG index, Coleman Liau Index, Dale Chall Readability Score]

In [268]:
# Facet by category and metric
plot_metrics = ["flesch_reading_ease", "smog_index", "coleman_liau_index", "dale_chall_readability_score"]
fig = px.box(
    to_plot[to_plot["metric"].isin(plot_metrics)], 
    # to_plot[to_plot["category"].isin(["Data", "Output"])], 
    x = "Score", color = "Respondent", points = "all", 
    facet_row = "category", facet_col = "metric", 
    title = "Readability Metrics by Category: Human vs LLM Output",
    color_discrete_map = colour_mapping,
    height = 1200, width = 1200
#  , template = "simple_white"
    ).update_layout(title_x = 0.25).update_xaxes(matches = None).update_yaxes(tickangle = 45)

for annotation in fig["layout"]["annotations"]: 
    annotation["textangle"] = 0

fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[1].replace("_", " ").title()))
fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels = True))
fig.show()

In [269]:
# Facet only by metric
fig = px.violin(
    to_plot[to_plot["metric"].isin(plot_metrics)], 
    x = "Score", color = "Respondent", 
    box = True, points = "all",
    facet_col = "metric", 
    title = "Overall Readability Metrics: Human vs LLM Output",
    color_discrete_map = colour_mapping,
    height = 600, width = 1200,
    ).update_xaxes(matches = None).update_layout(title_x = 0.5)

# fig = px.box(to_plot, x = "Score", color = "Respondent", 
#              points = "all",
#              facet_col = "metric", 
#              height = 500, width = 1300
#              ).update_xaxes(matches = None).update_layout(boxmode = "group", boxgap = 0, boxgroupgap = 0.5)

fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[1].replace("_", " ").title()))
fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels = True))
fig.show()

In [ ]:
plot_metrics = [
    "smog_index",
    "coleman_liau_index",
    "dale_chall_readability_score",
    "flesch_kincaid_grade",
    "automated_readability_index",
    "linsear_write_formula",
    "gunning_fog"
    ]

px.line_polar(
    to_plot[to_plot["metric"].isin(plot_metrics)].groupby(["Respondent", "metric"])["Score"].mean().reset_index(), 
    r = "Score", theta = "metric", line_close = True, color = "Respondent")

##### Similarity

In [305]:
list(xaiqb_human["questions"][category].keys())

['What are the affordances of the system?', 'What does this UI element do?']

In [308]:
questions = []
for category in xaiqb_human["questions"].keys():
    for question in xaiqb_human["questions"][category].keys():
        questions += [question]

questions

['What kind of data was the system trained on?',
 'What is the source of the training data?',
 'How were the labels/ground-truth produced?',
 'What is the sample size of the training data?',
 'What dataset(s) is the system NOT using?',
 'What are the potential limitations/biases of the data?',
 'What kind of output does the system give?',
 'What does the system output mean?',
 "What is the scope of the system's capability? Can it do...?",
 'How is the output used for other system component(s)?',
 'How should I best utilize the output of the system?',
 'How should the output fit in my workflow?',
 'What is the source of the information object(s)?',
 'What is the scope of the output data?',
 'What is the amount of the output data?',
 'What logic is used for the initial output/recommendation?',
 'How accurate/precise/reliable are the predictions?',
 'How often does the system make mistakes?',
 'In what situations is the system likely to be correct/incorrect?',
 'What are the limitations o

https://spotintelligence.com/2022/12/19/text-similarity-python/#1_Text_similarity_with_NLTK

In [271]:
# def pull_similarity_scores(responses1, responses2 = xaiqb_human, suffix = "", scores_only = True):
#     similarity_scores = []

#     for category in responses1["questions"].keys():    
#         for question in responses1["questions"][category]["question_answer"].keys():
            
#             q_1 = responses1["questions"][category]["question_answer"][question]["question_interpretation"]
#             q_2 = responses2["questions"][category]["question_answer"][question]["question_interpretation"]
            
#             ans_1 = responses1["questions"][category]["question_answer"][question]["answer"]
#             ans_2 = responses2["questions"][category]["question_answer"][question]["answer"]
            
#             # Tokenize and encode the texts
#             q_encoding1 = model.encode(q_1, normalize_embeddings = True)
#             q_encoding2 = model.encode(q_2, normalize_embeddings = True)

#             ans_encoding_1 = model.encode(ans_1, normalize_embeddings = True)
#             ans_encoding_2 = model.encode(ans_2, normalize_embeddings = True)
#             # Calculate the cosine similarity between the embeddings
#             q_similarity = np.dot(q_encoding1, q_encoding2) / (np.linalg.norm(q_encoding1) * np.linalg.norm(q_encoding2))
#             answer_similarity = np.dot(ans_encoding_1, ans_encoding_2) / (np.linalg.norm(ans_encoding_1) * np.linalg.norm(ans_encoding_2))
#             # question_similarity = np.dot(question_encoding, llm_encoding) / (np.linalg.norm(question_encoding) * np.linalg.norm(llm_encoding))
            
#             cols = [x + f" {suffix}" for x in ["Answer Similarity", "Question Similarity"]]
#             similarity_scores += [pd.DataFrame({
#                 "Category": category,
#                 "Question": question,
#                 cols[0]: [answer_similarity],
#                 cols[1]: [q_similarity]
#                 # "Question Similarity": [question_similarity]
#             })]

#     similarity_scores = (pd.concat(similarity_scores)
#                         #  .sort_values("Answer Similarity", ascending = False)
#                         #  .reset_index(drop = True)
#                          .round(3)
#                          )
#     return similarity_scores[cols] if scores_only else similarity_scores

In [272]:
# Human x ChatGPT
# Human x Gemini
# Human x Claude

# ChatGPT x Gemini
# ChatGPT x Claude

# Gemini x Claude

In [275]:
def pull_similariy_scores(model):
    similarity_scores = []
    for category in xaiqb_human["questions"].keys():  
        for question in xaiqb_human["questions"][category].keys():

            answers = [
                xaiqb_human["questions"][category][question]["answer"],
                xaiqb_claude["questions"][category][question]["answer"],
                xaiqb_chatgpt["questions"][category][question]["answer"],
                xaiqb_gemini["questions"][category][question]["answer"]
                ]
            
            question_interpretations = [
                xaiqb_human["questions"][category][question]["question_interpretation"],
                xaiqb_claude["questions"][category][question]["question_interpretation"],
                xaiqb_chatgpt["questions"][category][question]["question_interpretation"],
                xaiqb_gemini["questions"][category][question]["question_interpretation"]
                ]
            
            answer_encoding = model.encode(answers)

            human_answer_cosine_similarity      = util.cos_sim(answer_encoding[0], answer_encoding[1:]).tolist()[0]
            claude_answer_cosine_similarity     = util.cos_sim(answer_encoding[1], answer_encoding[2:]).tolist()[0]
            chatgpt_answer_cosine_similarity    = util.cos_sim(answer_encoding[2], answer_encoding[3:]).tolist()[0]

            question_interpretation_encoding = model.encode(question_interpretations)

            human_question_interpretation_cosine_similarity     = util.cos_sim(question_interpretation_encoding[0], question_interpretation_encoding[1:]).tolist()[0]
            claude_question_interpretation_cosine_similarity    = util.cos_sim(question_interpretation_encoding[1], question_interpretation_encoding[2:]).tolist()[0]
            chatgpt_question_interpretation_cosine_similarity   = util.cos_sim(question_interpretation_encoding[2], question_interpretation_encoding[3:]).tolist()[0]

            similarity_scores += [pd.DataFrame({
                "Category": category,
                "Question": question,

                "Answer Human Claude"    : [human_answer_cosine_similarity[0]],
                "Answer Human ChatGPT"   : [human_answer_cosine_similarity[1]],
                "Answer Human Gemini"    : [human_answer_cosine_similarity[2]],
                "Answer Claude ChatGPT"  : [claude_answer_cosine_similarity[0]],
                "Answer Claude Gemini"   : [claude_answer_cosine_similarity[1]],
                "Answer ChatGPT Gemini"  : [chatgpt_answer_cosine_similarity[0]],

                "Question Human Claude"    : [human_question_interpretation_cosine_similarity[0]],
                "Question Human ChatGPT"   : [human_question_interpretation_cosine_similarity[1]],
                "Question Human Gemini"    : [human_question_interpretation_cosine_similarity[2]],
                "Question Claude ChatGPT"  : [claude_question_interpretation_cosine_similarity[0]],
                "Question Claude Gemini"   : [claude_question_interpretation_cosine_similarity[1]],
                "Question ChatGPT Gemini"  : [chatgpt_question_interpretation_cosine_similarity[0]],

                })
                ]
    
    similarity_scores_df = pd.concat(similarity_scores)
    # columns = [("", "Category"),("", "Question")] + [("Answer Similarity", x) for x in ["Human Claude", "Human ChatGPT", "Human Gemini", "Claude ChatGPT", "Claude Gemini", "ChatGPT Gemini"]] + [("Question Interpretation Similarity", x) for x in ["Human Claude", "Human ChatGPT"]]#, "Human Gemini", "Claude ChatGPT", "Claude Gemini", "ChatGPT Gemini"]]
    # similariy_scores_df.columns = pd.MultiIndex.from_tuples(columns)
    return similarity_scores_df

In [276]:
# all-mpnet-base-v2, all-MiniLM-L12-v2, all-MiniLM-L6-v2
similarity_scores = pull_similariy_scores(SentenceTransformer("all-MiniLM-L6-v2"))
similarity_scores.head()

,Category,Question,Answer Human Claude,Answer Human ChatGPT,Answer Human Gemini,Answer Claude ChatGPT,Answer Claude Gemini,Answer ChatGPT Gemini,Question Human Claude,Question Human ChatGPT,Question Human Gemini,Question Claude ChatGPT,Question Claude Gemini,Question ChatGPT Gemini
0,Data,What kind of data was the system trained on?,0.691934,0.633412,0.672870,0.848518,0.824950,0.797864,0.377013,0.367805,0.590757,0.500413,0.631572,0.457110
0,Data,What is the source of the training data?,0.930262,0.859344,0.404775,0.928266,0.430799,0.352441,0.665581,0.691554,0.700646,0.652804,0.805064,0.744669
0,Data,How were the labels/ground-truth produced?,0.763692,0.695859,0.616209,0.917364,0.774350,0.723382,0.222053,0.363766,0.255966,0.355275,0.861112,0.296252
0,Data,What is the sample size of the training data?,0.738893,0.510891,0.466639,0.641991,0.507585,0.379175,0.573734,0.390406,0.776522,0.508948,0.672545,0.613957
0,Data,What dataset(s) is the system NOT using?,0.671490,0.662503,0.575520,0.672425,0.550025,0.793000,0.535952,0.544536,0.430949,0.702646,0.614216,0.412155


In [ ]:
# similarity_scores.to_clipboard(index = False)

In [284]:
import itertools, re
answer_metrics = [f"Answer {x} {y}" for x, y in itertools.combinations(["Human", "Claude", "ChatGPT", "Gemini"], 2)]
question_metrics = [f"Question {x} {y}" for x, y in itertools.combinations(["Human", "Claude", "ChatGPT", "Gemini"], 2)]


In [283]:
to_plot = similarity_scores.groupby("Category").mean(numeric_only = True).reset_index().melt(id_vars = "Category")

In [285]:
to_plot[to_plot["variable"].apply(lambda x: re.search("Answer Human", x) is not None)]

,Category,variable,value
0,Data,Answer Human Claude,0.743570
1,How (global model-wide explanation),Answer Human Claude,0.618163
2,Others,Answer Human Claude,0.522952
3,Output,Answer Human Claude,0.569730
4,Performance,Answer Human Claude,0.598300
5,UI,Answer Human Claude,0.583885
6,Data,Answer Human ChatGPT,0.641748
7,How (global model-wide explanation),Answer Human ChatGPT,0.541211
8,Others,Answer Human ChatGPT,0.384738
9,Output,Answer Human ChatGPT,0.511804


In [279]:
px.line_polar(to_plot[to_plot["variable"].apply(lambda x: re.search("Answer Human", x) is not None)], 
              r = "value", range_r = [0, 1], theta = "Category", color = "variable", line_close = True, 
              height = 500, width = 1000,
              title = "Semantic Similarity Scores of Human vs LLM Generated Answers").update_layout(legend = dict(orientation = "h"), title_x = 0.5)

In [280]:
px.line_polar(to_plot[to_plot["variable"].apply(lambda x: re.search("Question Human", x) is not None)], 
              r = "value", range_r = [0, 1], theta = "Category", color = "variable", line_close = True, 
              height = 500, width = 1000,
              title = "Semantic Similarity Scores of Human vs LLM Interpretations of Questions").update_layout(legend = dict(orientation = "h"), title_x = 0.5)

In [ ]:
for i in range(len(answer_metrics)):
    print(f"{answer_metrics[i]}: {similarity_scores[[answer_metrics[1], question_metrics[i]]].corr().iloc[0, 1]:.3f}")

Answer Human Claude: 0.008
Answer Human ChatGPT: 0.202
Answer Human Gemini: -0.060
Answer Claude ChatGPT: -0.102
Answer Claude Gemini: 0.326
Answer ChatGPT Gemini: 0.264


In [ ]:
largest_similarity = similarity_scores.nlargest(3, "similarity").reset_index(drop = True)
largest_similarity

,category,question,similarity
0,Data,How were the labels/ground-truth produced?,0.757691
1,Performance,What kind of mistakes is the system likely to ...,0.720586
2,How (global model-wide explanation),What are the top rules/features that determine...,0.719658


In [ ]:
largest_similarity = similarity_scores.nlargest(3, "similarity").reset_index(drop = True)
largest_similarity

,category,question,similarity
0,How (global model-wide explanation),What are the top rules/features that determine...,0.825490
1,Data,What is the source of the training data?,0.813104
2,Performance,What kind of mistakes is the system likely to ...,0.799797


In [ ]:
for category, question, similarity in largest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human[category]["question_answer"][question]["answer"]}
    LLM:\t{xaiqb_gemini[category]["question_answer"][question]["answer"]}
    ''')


    Data | How were the labels/ground-truth produced?
    Similarity score: 0.758
    --------------------------------------------------------------------------------
    Human:	Respondants were asked if they had recieved the H1N1 vaccination or not. This is a binary 0/1 feature indicating no/yes respectively.
    LLM:	The 'ground truth' for our model is whether a person received the H1N1 vaccine. This information was self-reported by the survey respondents.
    

    Performance | What kind of mistakes is the system likely to make?
    Similarity score: 0.721
    --------------------------------------------------------------------------------
    Human:	The system will incorrectly identify if a person gets the vaccine or not (i.e. False Positive or False Negative).
    LLM:	The most common mistake is a 'false positive', where the model predicts an individual will get vaccinated, but they do not. It is less likely to make a 'false negative' error, where it predicts someone will not ge

In [ ]:
for category, question, similarity in largest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human["Global"][category]["question_answer"][question]}
    LLM:\t{xaiqb_gemini["Global"][category]["question_answer"][question]}
    ''')


    How (global model-wide explanation) | What are the top rules/features that determine its predictions?
    Similarity score: 0.825
    --------------------------------------------------------------------------------
    Human:	Top features are if the doctor reccomends the H1N1 vaccine, if they believe the vaccine is effective, and how much of a risk they believe H1N1 is.
    LLM:	The top three most influential features are whether a doctor recommended the H1N1 vaccine, the respondent's perceived risk of getting sick from H1N1, and their opinion on the vaccine's effectiveness.【2-34】
    

    Data | What is the source of the training data?
    Similarity score: 0.813
    --------------------------------------------------------------------------------
    Human:	Data comes from the National 2009 H1N1 Flu Survey (NHFS).
    LLM:	The data comes from the 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition on DrivenData, which is based on the National 2009 H1N1 Flu Su

In [ ]:
smallest_similarity = similarity_scores.nsmallest(3, "similarity").reset_index(drop = True)
smallest_similarity

,category,question,similarity
0,How (global model-wide explanation),How does it weigh different features?,0.072991
1,Output,What is the scope of the output data?,0.188284
2,Performance,What are the limitations of the system?,0.218875


In [ ]:
smallest_similarity = similarity_scores.nsmallest(3, "similarity").reset_index(drop = True)
smallest_similarity

,category,question,similarity
0,Data,What dataset(s) is the system NOT using?,0.097394
1,Data,What are the potential limitations/biases of t...,0.118485
2,Output,What is the scope of the output data?,0.129578


In [ ]:
for category, question, similarity in smallest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human[category]["question_answer"][question]["answer"]}
    LLM:\t{xaiqb_gemini[category]["question_answer"][question]["answer"]}
    ''')


    How (global model-wide explanation) | How does it weigh different features?
    Similarity score: 0.073
    --------------------------------------------------------------------------------
    Human:	Feature weights are not assigned when using a RandomForestClassifier(). Instead, Gini (impurity) is used to measure the quality of a split in a decision tree, which is how 'mixed' the classes are in a node. Lower impurity means the node is more pure (mostly one class). For a binary classifier, this impurity is measured from [0, 0.5].
    LLM:	The most influential factor is whether a doctor recommended the H1N1 vaccine. Opinions about the vaccine's effectiveness and the risk of H1N1 infection are also highly weighted. Demographic factors like age and race have a smaller, but still significant, influence.
    

    Output | What is the scope of the output data?
    Similarity score: 0.188
    --------------------------------------------------------------------------------
    Human:	The

In [ ]:
for category, question, similarity in smallest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human["Global"][category]["question_answer"][question]}
    LLM:\t{xaiqb_gemini["Global"][category]["question_answer"][question]}
    ''')


    Data | What dataset(s) is the system NOT using?
    Similarity score: 0.097
    --------------------------------------------------------------------------------
    Human:	Only data provided by these two csv files are being used by the system.
    LLM:	The model is trained specifically on H1N1-related features and excludes features related to the seasonal flu vaccine to focus the prediction.【2-66】
    

    Data | What are the potential limitations/biases of the data?
    Similarity score: 0.118
    --------------------------------------------------------------------------------
    Human:	Limited transferability, as all respondants were persons 6 months or older living in the United States at the time of the interview, over telephone.
    LLM:	The dataset has a significant number of missing values in key fields like health insurance, employment industry, and occupation, which could introduce bias. For example, nearly 46% of respondents have missing health insurance information.【2

#### Readability Metric Descriptions

##### Flesch-Kincaid Grade Level
The U.S. Army uses Flesch-Kincaid Grade Level for assessing the difficulty of technical manuals. The commonwealth of Pennsylvania uses Flesch-Kincaid Grade Level for scoring automobile insurance policies to ensure their texts are no higher than a ninth grade level of reading difficulty. Many other U.S. states also use Flesch-Kincaid Grade Level to score other legal documents such as business policies and financial forms.

##### Flesch Reading Ease
The U.S. Department of Defense uses the Reading Ease test as the standard test of readability for its documents and forms. Florida requires that life insurance policies have a Flesch Reading Ease score of 45 or greater.

##### Dale Chall Readability
The Dale-Chall Formula is an accurate readability formula for the simple reason that it is based on the use of familiar words, rather than syllable or letter counts. Reading tests show that readers usually find it easier to read, process and recall a passage if they find the words familiar.

##### Automated Readability Index (ARI)
Unlike the other indices, the ARI, along with the Coleman-Liau, relies on a factor of characters per word, instead of the usual syllables per word. ARI is widely used on all types of texts.

##### Coleman Liau Index
The Coleman-Liau Formula usually gives a lower grade value than any of the Kincaid, ARI and Flesch values when applied to technical documents.

##### Gunning Fog
The Gunning fog index measures the readability of English writing. The index estimates the years of formal education needed to understand the text on a first reading. A fog index of 12 requires the reading level of a U.S. high school senior (around 18 years old).

##### SMOG
The SMOG Readability Formula (Simple Measure of Gobbledygook) is a popular method to use on health literacy materials.

##### SPACHE
The Spache Readability Formula is used for Primary-Grade Reading Materials, published in 1953 in The Elementary School Journal. The Spache Formula is best used to calculate the difficulty of text that falls at the 3rd grade level or below.

##### Linsear Write
Linsear Write is a readability metric for English text, purportedly developed for the United States Air Force to help them calculate the readability of their technical manuals.